<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Fundamental Unit:** One row represents the daily search performance of a single web page for a specific client (i.e., a content-day).

| Parameter | Specification |
|---|---|
| **Data Source** | `fact_content_daily_performance` from Hugging Face (~79 million rows) |
| **Granularity** | `report_date` + `client_hash_id` + `content_hash_id` |
| **Timeframe** | Late January 2025 through June 2026 (approx 17 months) |
| **Focus Month** | `2026-03` (Chosen from the middle of the dataset to prevent leaking the final month outcome) |
| **Feature Extraction** | The 30 days preceding the label window (January 30 to February 28) |
| **Label Definition** | The target month (March 2026). The model asks: did search impressions fall by 20% or more? |

**Why choose this granularity:** Since an SEO manager is deciding which *page* to fix, we need page-level data. The daily granularity allows us to calculate time-series momentum features that single snapshots miss. Our final scoring dataset rolls up these daily facts into a single row per page using the defined windows.

**Why a mid-panel month:** The final month in the dataset (June 2026) is the true holdout. Building our label logic on the final month would risk data leakage. March 2026 sits safely in the middle, providing ample history before and after.

In [1]:
import os, getpass, duckdb, pandas as pd
import numpy as np

# Authenticate Hugging Face
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF token: ')

con = duckdb.connect()
con.execute('INSTALL httpfs;')
con.execute('LOAD httpfs;')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Verify table bounds
print('--- Global Table Statistics ---')
full_info = con.sql(f"SELECT COUNT(*) AS total, MIN(report_date) as start_date, MAX(report_date) as end_date FROM {FACT_DAILY}").df()
print(f"Total rows: {full_info['total'][0]:,}")
print(f"Time span: {full_info['start_date'][0]} to {full_info['end_date'][0]}\n")

print('--- Analysis Month (March 2026) ---')
mar_info = con.sql(f"SELECT COUNT(*) AS march_rows, COUNT(DISTINCT content_hash_id) AS items, COUNT(DISTINCT client_hash_id) AS clients FROM {FACT_DAILY} WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'").df()
print(f"March rows: {mar_info['march_rows'][0]:,}")
print(f"Unique content pieces: {mar_info['items'][0]:,}")
print(f"Unique clients: {mar_info['clients'][0]}\n")

# Verify grain (should be zero violations)
grain_check = con.sql(f"SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c FROM {FACT_DAILY} WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' GROUP BY 1, 2, 3 HAVING COUNT(*) > 1").df()
print(f"Grain violations (duplicate rows for same day/content): {len(grain_check)}")

--- Global Table Statistics ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 78,835,655
Time span: 2025-01-27 00:00:00 to 2026-06-30 00:00:00

--- Analysis Month (March 2026) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 9,841,378
Unique content pieces: 331,437
Unique clients: 55



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (duplicate rows for same day/content): 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Column | Role | Type | Why |
|---|---|---|---|
| `imp_prev30` | Feature | Float | Total impressions in the 30 days before the prediction window. Strongest indicator of a page's historical value. |
| `clk_prev30` | Feature | Float | Total clicks in the pre-prediction window. |
| `pos_prev30` | Feature | Float | Average Google Search Console position. Helps identify if decay is due to rank drops. |
| `days_with_imp_prev30` | Feature | Int | Count of days with >0 impressions. Separates consistent performers from viral spikes. |
| `content_age_days` | Feature | Int | Days since publication. Older content naturally decays over time. |
| `imp_last30` | Denominator | Float | Used exclusively to calculate the label. Represents the traffic during the target month (March). |
| `is_declining` | Label | Binary | 1 if `imp_last30` < 0.8 * `imp_prev30`. This is the variable we are trying to score. |
| `client_hash_id` | Excluded | Hash | Anonymized identifier. Including it would cause the model to overfit to specific client behaviors rather than general decay patterns. |
| `content_hash_id` | Excluded | Hash | Same reason as above; it is merely an identifier. |

**Output structure:** The ultimate deliverable is a sorted queue of these content items, ordered by their likelihood of decline, evaluated using Precision@50.

In [2]:
print('--- Fact Table Schema ---')
print(con.sql(f"DESCRIBE SELECT * FROM {FACT_DAILY}").df().head(20).to_string(index=False))
print('\n--- Content Dimension Schema ---')
print(con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").df().to_string(index=False))

--- Fact Table Schema ---
             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  Non

In [3]:
%pip install -U duckdb --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import duckdb
import os
from dotenv import load_dotenv
load_dotenv()

print(duckdb.__version__)

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

hf_token = os.environ.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("--- FACT 1: Row Count and Date Span ---")
q1 = """
SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
display(con.execute(q1).df())

print("\n--- FACT 2: The Grain (Is it really Day x Client x Content?) ---")
q2 = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT report_date || client_hash_id || content_hash_id) as unique_grain_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
display(con.execute(q2).df())

print("\n--- FACT 3: Availability (Filtering with IS TRUE) ---")
q3 = """
SELECT COUNT(*) as rows_with_analytics
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE ga4_data_available IS TRUE
"""
display(con.execute(q3).df())

1.5.5
--- FACT 1: Row Count and Date Span ---


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31



--- FACT 2: The Grain (Is it really Day x Client x Content?) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_count
0,9841378,9841378



--- FACT 3: Availability (Filtering with IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_analytics
0,413966


In [5]:
# Missingness queries per column — the writing-data-contracts skill requires this
# Use the starter CSV since we're checking the 30k-row dataset

import pandas as pd

df_local = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print("=== MISSINGNESS PER COLUMN ===")
missing_report = pd.DataFrame({
    'column': df_local.columns,
    'missing_count': df_local.isna().sum().values,
    'missing_pct': (df_local.isna().mean().values * 100).round(1)
}).sort_values('missing_pct', ascending=False)

# Only show columns with missing data
missing_nonzero = missing_report[missing_report['missing_count'] > 0]
display(missing_nonzero)

print(f"\nTotal columns: {len(df_local.columns)}")
print(f"Columns with missing data: {len(missing_nonzero)}")

=== MISSINGNESS PER COLUMN ===


,column,missing_count,missing_pct
10,provider_used,21438,71.5
8,word_count,7699,25.7
9,char_count,7699,25.7
33,word_count_tier,7699,25.7
34,char_count_tier,7699,25.7
11,model_used,5733,19.1
43,trend_pct,3388,11.3
4,competition_level,2610,8.7
2,search_volume,2468,8.2
5,cpc,2468,8.2



Total columns: 44
Columns with missing data: 13


In [6]:
# Missingness by content_type — the skill says to check if missingness follows categories

print("=== MISSINGNESS BY CONTENT TYPE ===")
print("(A blind fillna(0) injects category signal — check this first)")
print()

key_cols = ['word_count', 'search_volume', 'competition', 'main_intent']
for col in key_cols:
    ct_missing = df_local.groupby('content_type')[col].apply(lambda x: x.isna().mean() * 100).round(1)
    print(f"{col} missing % by content_type:")
    print(ct_missing.to_string())
    print()

=== MISSINGNESS BY CONTENT TYPE ===
(A blind fillna(0) injects category signal — check this first)

word_count missing % by content_type:
content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3

search_volume missing % by content_type:
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4

competition missing % by content_type:
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4

main_intent missing % by content_type:
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.0



The Leakage Trap Experiment: To demonstrate leakage, I intentionally added trend_pct to my feature set. Because our target label (is_declining_label) is mathematically derived from the trend calculation, trend_pct is simply the answer in disguise.

When I included it, the model's Precision@50 spiked to an unrealistic near-perfect score. The tree simply split on trend_pct < 0 and ignored all actual SEO signals. I have now deleted trend_pct from the feature frame to keep the model honest.

In [7]:
# GA4 Availability Check
ga4_check = con.sql(f"SELECT ga4_data_available, COUNT(*) AS rows, COUNT(DISTINCT client_hash_id) as clients FROM {FACT_DAILY} WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' GROUP BY 1").df()
print('--- GA4 Data Flags in March 2026 ---')
print(ga4_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- GA4 Data Flags in March 2026 ---
 ga4_data_available    rows  clients
              False 6408671       43
               <NA> 3018741       22
               True  413966       41


In [8]:
# Building the safe features
features_df = con.sql(f"""
    WITH safe_agg AS (
        SELECT 
            f.content_hash_id,
            f.client_hash_id,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_avg_position END) AS pos_prev30,
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' AND f.gsc_impressions > 0 THEN f.report_date END) AS days_with_imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-01' AND f.report_date <= DATE '2026-03-31' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM {FACT_DAILY} f
        WHERE f.report_date >= DATE '2026-01-30' AND f.report_date <= DATE '2026-03-31'
          AND f.gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM safe_agg
""").df()

# Merge age
content_meta = con.sql(f"SELECT content_hash_id, DATEDIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days FROM {DIM_CONTENT}").df()
features_df = features_df.merge(content_meta, on='content_hash_id', how='left')

features_df['is_declining'] = (features_df['imp_last30'] < 0.8 * features_df['imp_prev30']).astype(int)

print(f'Total safe features extracted: {len(features_df):,}')
print(f'Decline rate in this extract: {features_df["is_declining"].mean():.3f}')
print('\nFeature Justifications:')
print('1. imp_prev30: Window closed Feb 28, completely knowable before March.')
print('2. clk_prev30: Closed window.')
print('3. pos_prev30: Closed window.')
print('4. days_with_imp_prev30: Only counts completed days prior to prediction.')
print('5. content_age_days: Fixed at publish date.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total safe features extracted: 81,521
Decline rate in this extract: 0.249

Feature Justifications:
1. imp_prev30: Window closed Feb 28, completely knowable before March.
2. clk_prev30: Closed window.
3. pos_prev30: Closed window.
4. days_with_imp_prev30: Only counts completed days prior to prediction.
5. content_age_days: Fixed at publish date.
